In [1]:
import random


# ============================================================
# Pretty Output Helpers (defined FIRST so they can be used anywhere)
# ============================================================
def print_banner(title):
    width = 60
    print("=" * width)
    print(title.center(width))
    print("=" * width)


# ============================================================
# Environment
# ============================================================
class GridEnvironment:
    def __init__(self, rows=4, cols=4, dirt_probability=0.5, seed=None):
        if seed is not None:
            random.seed(seed)
        self.rows, self.cols = rows, cols
        self.dirt = [
            [random.random() < dirt_probability for _ in range(cols)]
            for _ in range(rows)
        ]
        self.position = (0, 0)

    def get_percept(self):
        r, c = self.position
        return {"position": (r, c), "is_dirty": self.dirt[r][c]}

    def valid_moves(self):
        r, c = self.position
        moves = []
        if c < self.cols - 1:
            moves.append("Right")
        if c > 0:
            moves.append("Left")
        if r < self.rows - 1:
            moves.append("Down")
        if r > 0:
            moves.append("Up")
        return moves

    def apply_action(self, action):
        r, c = self.position
        if action == "Suck":
            self.dirt[r][c] = False
        elif action == "Right" and c < self.cols - 1:
            self.position = (r, c + 1)
        elif action == "Left" and c > 0:
            self.position = (r, c - 1)
        elif action == "Down" and r < self.rows - 1:
            self.position = (r + 1, c)
        elif action == "Up" and r > 0:
            self.position = (r - 1, c)

    def is_clean(self):
        return not any(any(row) for row in self.dirt)


# ============================================================
# Agent + Loop
# ============================================================
def simple_reflex_agent(percept, env):
    if percept["is_dirty"]:
        return "Suck"
    return random.choice(env.valid_moves())


def run_agent(env, decide, max_steps=200):
    steps = 0
    for _ in range(max_steps):
        percept = env.get_percept()
        if env.is_clean():
            break
        action = decide(percept, env)
        env.apply_action(action)
        steps += 1
    return steps


# ============================================================
# Main Run
# ============================================================
if __name__ == "__main__":
    print_banner("LAB TASK 1: SIMPLE REFLEX AGENT")

    env = GridEnvironment(rows=4, cols=4, dirt_probability=0.5, seed=11)
    steps = run_agent(env, simple_reflex_agent)

    print("\n--- CONFIGURATION ---")
    print("  Grid size        : 4 x 4")
    print("  Dirt probability : 0.5")
    print("  Seed             : 11")

    print("\n--- RESULT ---")
    print(f"  Simple reflex (random) steps: {steps}")

              LAB TASK 1: SIMPLE REFLEX AGENT               

--- CONFIGURATION ---
  Grid size        : 4 x 4
  Dirt probability : 0.5
  Seed             : 11

--- RESULT ---
  Simple reflex (random) steps: 48


In [2]:
import random
import copy


# ============================================================
# Pretty Output Helpers
# ============================================================
def print_banner(title):
    width = 60
    print("=" * width)
    print(title.center(width))
    print("=" * width)


# ============================================================
# Environment
# ============================================================
class GridEnvironment:
    def __init__(self, rows=4, cols=4, dirt_probability=0.5, seed=None):
        if seed is not None:
            random.seed(seed)
        self.rows, self.cols = rows, cols
        self.dirt = [
            [random.random() < dirt_probability for _ in range(cols)]
            for _ in range(rows)
        ]
        self.position = (0, 0)

    def get_percept(self):
        r, c = self.position
        return {"position": (r, c), "is_dirty": self.dirt[r][c]}

    def valid_moves(self):
        r, c = self.position
        moves = []
        if c < self.cols - 1:
            moves.append("Right")
        if c > 0:
            moves.append("Left")
        if r < self.rows - 1:
            moves.append("Down")
        if r > 0:
            moves.append("Up")
        return moves

    def apply_action(self, action):
        r, c = self.position
        if action == "Suck":
            self.dirt[r][c] = False
        elif action == "Right" and c < self.cols - 1:
            self.position = (r, c + 1)
        elif action == "Left" and c > 0:
            self.position = (r, c - 1)
        elif action == "Down" and r < self.rows - 1:
            self.position = (r + 1, c)
        elif action == "Up" and r > 0:
            self.position = (r - 1, c)

    def is_clean(self):
        return not any(any(row) for row in self.dirt)


# ============================================================
# Agents + Loop
# ============================================================
def simple_reflex_agent(percept, env):
    if percept["is_dirty"]:
        return "Suck"
    return random.choice(env.valid_moves())


def run_agent(env, decide, max_steps=200):
    steps = 0
    for _ in range(max_steps):
        percept = env.get_percept()
        if env.is_clean():
            break
        action = decide(percept, env)
        env.apply_action(action)
        steps += 1
    return steps


def neighbor_position(pos, action):
    r, c = pos
    return {
        "Right": (r, c + 1),
        "Left": (r, c - 1),
        "Down": (r + 1, c),
        "Up": (r - 1, c),
    }[action]


class ModelBasedAgent:
    def __init__(self):
        self.visited = set()

    def act(self, percept, env):
        self.visited.add(percept["position"])
        if percept["is_dirty"]:
            return "Suck"
        moves = env.valid_moves()
        here = percept["position"]
        unvisited = [m for m in moves
                     if neighbor_position(here, m) not in self.visited]
        if unvisited:
            return random.choice(unvisited)
        return random.choice(moves)


# ============================================================
# Main Run
# ============================================================
if __name__ == "__main__":
    print_banner("LAB TASK 2: MODEL-BASED AGENT")

    # Same starting layout for both agents
    random.seed(11)
    env1 = GridEnvironment(rows=4, cols=4, dirt_probability=0.5)
    dirt_copy = copy.deepcopy(env1.dirt)
    steps1 = run_agent(env1, simple_reflex_agent)

    random.seed(11)
    env2 = GridEnvironment(rows=4, cols=4, dirt_probability=0.5)
    env2.dirt = copy.deepcopy(dirt_copy)
    agent2 = ModelBasedAgent()
    steps2 = run_agent(env2, agent2.act)

    print("\n--- CONFIGURATION ---")
    print("  Grid size : 4 x 4")
    print("  Seed      : 11")
    print("  Both agents started on the SAME dirt layout")

    print("\n--- RESULT ---")
    print(f"  Simple reflex (random) steps : {steps1}")
    print(f"  Model-based steps            : {steps2}")
    print(f"  Difference                   : {steps1 - steps2}")

               LAB TASK 2: MODEL-BASED AGENT                

--- CONFIGURATION ---
  Grid size : 4 x 4
  Seed      : 11
  Both agents started on the SAME dirt layout

--- RESULT ---
  Simple reflex (random) steps : 48
  Model-based steps            : 42
  Difference                   : 6


In [3]:
import random


# ============================================================
# Pretty Output Helpers
# ============================================================
def print_banner(title):
    width = 60
    print("=" * width)
    print(title.center(width))
    print("=" * width)


# ============================================================
# Environment
# ============================================================
class GridEnvironment:
    def __init__(self, rows=4, cols=4, dirt_probability=0.5, seed=None):
        if seed is not None:
            random.seed(seed)
        self.rows, self.cols = rows, cols
        self.dirt = [
            [random.random() < dirt_probability for _ in range(cols)]
            for _ in range(rows)
        ]
        self.position = (0, 0)

    def get_percept(self):
        r, c = self.position
        return {"position": (r, c), "is_dirty": self.dirt[r][c]}

    def valid_moves(self):
        r, c = self.position
        moves = []
        if c < self.cols - 1:
            moves.append("Right")
        if c > 0:
            moves.append("Left")
        if r < self.rows - 1:
            moves.append("Down")
        if r > 0:
            moves.append("Up")
        return moves

    def apply_action(self, action):
        r, c = self.position
        if action == "Suck":
            self.dirt[r][c] = False
        elif action == "Right" and c < self.cols - 1:
            self.position = (r, c + 1)
        elif action == "Left" and c > 0:
            self.position = (r, c - 1)
        elif action == "Down" and r < self.rows - 1:
            self.position = (r + 1, c)
        elif action == "Up" and r > 0:
            self.position = (r - 1, c)

    def is_clean(self):
        return not any(any(row) for row in self.dirt)


# ============================================================
# Goal-Based Agent
# ============================================================
def goal_based_agent(position, goal):
    r, c = position
    gr, gc = goal
    if r < gr:
        return "Down"
    if r > gr:
        return "Up"
    if c < gc:
        return "Right"
    if c > gc:
        return "Left"
    return "NoOp"


def run_goal(env, goal, label):
    print(f"\n--- {label} ---")
    print(f"  Start : {env.position}")
    print(f"  Goal  : {goal}")
    for step in range(20):
        if env.position == goal:
            print(f"  Reached the goal in {step} steps.")
            break
        action = goal_based_agent(env.position, goal)
        env.apply_action(action)
        print(f"  Step {step+1}: moved {action:<6} -> now at {env.position}")


# ============================================================
# Main Run
# ============================================================
if __name__ == "__main__":
    print_banner("LAB TASK 3: GOAL-BASED AGENT")

    env1 = GridEnvironment(rows=4, cols=4)
    env1.position = (0, 0)
    run_goal(env1, (3, 2), "RUN 1")

    env2 = GridEnvironment(rows=4, cols=4)
    env2.position = (3, 1)
    run_goal(env2, (0, 3), "RUN 2")

                LAB TASK 3: GOAL-BASED AGENT                

--- RUN 1 ---
  Start : (0, 0)
  Goal  : (3, 2)
  Step 1: moved Down   -> now at (1, 0)
  Step 2: moved Down   -> now at (2, 0)
  Step 3: moved Down   -> now at (3, 0)
  Step 4: moved Right  -> now at (3, 1)
  Step 5: moved Right  -> now at (3, 2)
  Reached the goal in 5 steps.

--- RUN 2 ---
  Start : (3, 1)
  Goal  : (0, 3)
  Step 1: moved Up     -> now at (2, 1)
  Step 2: moved Up     -> now at (1, 1)
  Step 3: moved Up     -> now at (0, 1)
  Step 4: moved Right  -> now at (0, 2)
  Step 5: moved Right  -> now at (0, 3)
  Reached the goal in 5 steps.


# Lab Task 4 — Analysis: Classify Your Own Agent

**No new code.** This analysis uses the Task 1 (simple reflex) and Task 2 (model-based) agents and the 2D Grid Vacuum World environment.

---

## 1. The Six Building Blocks

| Building Block | Your Task 1/2 Agent, in Your Own Words |
|---|---|
| **Sensors** | `get_percept()` returns the agent's current `(row, col)` position and whether that cell `is_dirty`. Only the current cell is perceived. |
| **Actuators** | `apply_action()` performs `Suck`, `Up`, `Down`, `Left`, `Right` — changing dirt or moving the agent. |
| **Goal** | Clean every room in the grid. |
| **State** | The full 2D dirt grid plus the agent's position. |
| **Observation** | A dict: `{"position": (r, c), "is_dirty": bool}` — a partial view of the state. |
| **Action** | `Suck` if dirty, otherwise a random valid move (Task 1) or an unvisited neighbour (Task 2). |

---

## 2. Environment Observability

**Partially observable.**

The percept returned by `get_percept()` only contains the agent's own `(row, col)` position and whether the **current** cell `is_dirty`. The agent cannot see the dirt status of any other room, nor the full 2D grid. Because the observation does not reveal the complete state of the world, the environment is partially observable.

---

## 3. Environment Static vs Dynamic

**Static.**

Nothing in the world changes while the agent is not acting: dirt does not reappear, no other agents move, and the grid layout stays fixed. Only the agent's own actions change the state. Because the environment does not change on its own between actions, it is static.

---

## 4. Autonomy Ladder Placement

**Task 1 agent (simple reflex, no memory):** sits at **"Acts and Reports"**. It decides and acts on its own using only the current percept, and simply reports its actions through the step log.

**Task 2 agent (model-based, with memory):** sits closer to **"Acts Alone"**. It maintains an internal model (visited positions), uses that memory to choose smarter moves without human input, and completes the task autonomously.

---

## 5. Summary Table

| Question | Answer |
|---|---|
| Fully or partially observable? | **Partially observable** — percept only shows the current cell. |
| Static or dynamic? | **Static** — nothing changes while the agent is idle. |
| Autonomy level (Task 1)? | **Acts and Reports** — reacts to current percept, logs actions. |
| Autonomy level (Task 2)? | **Acts Alone** — uses memory to act autonomously. |

In [4]:
import random


# ============================================================
# Pretty Output Helpers
# ============================================================
def print_banner(title):
    width = 60
    print("=" * width)
    print(title.center(width))
    print("=" * width)


# ============================================================
# Environment
# ============================================================
class GridEnvironment:
    def __init__(self, rows=4, cols=4, dirt_probability=0.5, seed=None):
        if seed is not None:
            random.seed(seed)
        self.rows, self.cols = rows, cols
        self.dirt = [
            [random.random() < dirt_probability for _ in range(cols)]
            for _ in range(rows)
        ]
        self.position = (0, 0)

    def get_percept(self):
        r, c = self.position
        return {"position": (r, c), "is_dirty": self.dirt[r][c]}

    def valid_moves(self):
        r, c = self.position
        moves = []
        if c < self.cols - 1:
            moves.append("Right")
        if c > 0:
            moves.append("Left")
        if r < self.rows - 1:
            moves.append("Down")
        if r > 0:
            moves.append("Up")
        return moves

    def apply_action(self, action):
        r, c = self.position
        if action == "Suck":
            self.dirt[r][c] = False
        elif action == "Right" and c < self.cols - 1:
            self.position = (r, c + 1)
        elif action == "Left" and c > 0:
            self.position = (r, c - 1)
        elif action == "Down" and r < self.rows - 1:
            self.position = (r + 1, c)
        elif action == "Up" and r > 0:
            self.position = (r - 1, c)

    def is_clean(self):
        return not any(any(row) for row in self.dirt)


# ============================================================
# Agents
# ============================================================
def simple_reflex_agent(percept, env):
    if percept["is_dirty"]:
        return "Suck"
    return random.choice(env.valid_moves())


def neighbor_position(pos, action):
    r, c = pos
    return {
        "Right": (r, c + 1),
        "Left": (r, c - 1),
        "Down": (r + 1, c),
        "Up": (r - 1, c),
    }[action]


class ModelBasedAgent:
    def __init__(self):
        self.visited = set()

    def act(self, percept, env):
        self.visited.add(percept["position"])
        if percept["is_dirty"]:
            return "Suck"
        moves = env.valid_moves()
        here = percept["position"]
        unvisited = [m for m in moves
                     if neighbor_position(here, m) not in self.visited]
        if unvisited:
            return random.choice(unvisited)
        return random.choice(moves)


# ============================================================
# Loop + Fair Comparison
# ============================================================
def run_agent(env, decide, max_steps=200):
    steps = 0
    for _ in range(max_steps):
        percept = env.get_percept()
        if env.is_clean():
            break
        action = decide(percept, env)
        env.apply_action(action)
        steps += 1
    return steps


def steps_per_layout(decide_factory, n_layouts=30, rows=4, cols=4,
                     dirt_probability=0.5, max_steps=200, seed_base=1000):
    total = 0
    for i in range(n_layouts):
        env = GridEnvironment(rows=rows, cols=cols,
                              dirt_probability=dirt_probability,
                              seed=seed_base + i)
        decide = decide_factory()
        total += run_agent(env, decide, max_steps=max_steps)
    return total / n_layouts


def reflex_factory():
    return lambda percept, env: simple_reflex_agent(percept, env)


def model_factory():
    agent = ModelBasedAgent()
    return agent.act


# ============================================================
# Main Run
# ============================================================
if __name__ == "__main__":
    print_banner("LAB TASK 5: FAIR COMPARISON")

    avg_reflex = steps_per_layout(reflex_factory, n_layouts=30)
    avg_model = steps_per_layout(model_factory, n_layouts=30)

    print("\n--- CONFIGURATION ---")
    print("  Layouts tested per agent : 30")
    print("  Seeds used               : 1000 .. 1029")
    print("  Grid size                : 4 x 4")

    print("\n--- AVERAGE STEPS PER LAYOUT ---")
    print(f"  Simple reflex (random) : {avg_reflex:.2f}")
    print(f"  Model-based            : {avg_model:.2f}")
    print(f"  Difference             : {avg_reflex - avg_model:.2f}")

                LAB TASK 5: FAIR COMPARISON                 

--- CONFIGURATION ---
  Layouts tested per agent : 30
  Seeds used               : 1000 .. 1029
  Grid size                : 4 x 4

--- AVERAGE STEPS PER LAYOUT ---
  Simple reflex (random) : 75.93
  Model-based            : 29.87
  Difference             : 46.07


In [5]:
import random


# ============================================================
# Pretty Output Helpers
# ============================================================
def print_banner(title):
    width = 60
    print("=" * width)
    print(title.center(width))
    print("=" * width)


# ============================================================
# Environment + Sensor-Upgraded Environment
# ============================================================
class GridEnvironment:
    def __init__(self, rows=4, cols=4, dirt_probability=0.5, seed=None):
        if seed is not None:
            random.seed(seed)
        self.rows, self.cols = rows, cols
        self.dirt = [
            [random.random() < dirt_probability for _ in range(cols)]
            for _ in range(rows)
        ]
        self.position = (0, 0)

    def get_percept(self):
        r, c = self.position
        return {"position": (r, c), "is_dirty": self.dirt[r][c]}

    def valid_moves(self):
        r, c = self.position
        moves = []
        if c < self.cols - 1:
            moves.append("Right")
        if c > 0:
            moves.append("Left")
        if r < self.rows - 1:
            moves.append("Down")
        if r > 0:
            moves.append("Up")
        return moves

    def apply_action(self, action):
        r, c = self.position
        if action == "Suck":
            self.dirt[r][c] = False
        elif action == "Right" and c < self.cols - 1:
            self.position = (r, c + 1)
        elif action == "Left" and c > 0:
            self.position = (r, c - 1)
        elif action == "Down" and r < self.rows - 1:
            self.position = (r + 1, c)
        elif action == "Up" and r > 0:
            self.position = (r - 1, c)

    def is_clean(self):
        return not any(any(row) for row in self.dirt)


class NeighborSensorGridEnvironment(GridEnvironment):
    def get_percept(self):
        r, c = self.position
        neighbours = {}
        for action, (nr, nc) in {
            "Right": (r, c + 1),
            "Left": (r, c - 1),
            "Down": (r + 1, c),
            "Up": (r - 1, c),
        }.items():
            if 0 <= nr < self.rows and 0 <= nc < self.cols:
                neighbours[action] = self.dirt[nr][nc]
        return {
            "position": (r, c),
            "is_dirty": self.dirt[r][c],
            "neighbours_dirty": neighbours,
        }


# ============================================================
# Agents
# ============================================================
def simple_reflex_agent(percept, env):
    if percept["is_dirty"]:
        return "Suck"
    return random.choice(env.valid_moves())


def sensor_reflex_agent(percept, env):
    if percept["is_dirty"]:
        return "Suck"
    dirty_neighbours = [a for a, d in percept["neighbours_dirty"].items() if d]
    if dirty_neighbours:
        return random.choice(dirty_neighbours)
    return random.choice(env.valid_moves())


# ============================================================
# Loop + Fair Comparison
# ============================================================
def run_agent(env, decide, max_steps=200):
    steps = 0
    for _ in range(max_steps):
        percept = env.get_percept()
        if env.is_clean():
            break
        action = decide(percept, env)
        env.apply_action(action)
        steps += 1
    return steps


def steps_per_layout(env_class, decide_factory, n_layouts=30, rows=4, cols=4,
                     dirt_probability=0.5, max_steps=200, seed_base=1000):
    total = 0
    for i in range(n_layouts):
        env = env_class(rows=rows, cols=cols,
                        dirt_probability=dirt_probability,
                        seed=seed_base + i)
        decide = decide_factory()
        total += run_agent(env, decide, max_steps=max_steps)
    return total / n_layouts


def reflex_factory():
    return lambda percept, env: simple_reflex_agent(percept, env)


def sensor_factory():
    return lambda percept, env: sensor_reflex_agent(percept, env)


# ============================================================
# Main Run
# ============================================================
if __name__ == "__main__":
    print_banner("LAB TASK 6: SENSOR UPGRADE")

    avg_reflex = steps_per_layout(GridEnvironment, reflex_factory, n_layouts=30)
    avg_sensor = steps_per_layout(NeighborSensorGridEnvironment, sensor_factory, n_layouts=30)

    print("\n--- CONFIGURATION ---")
    print("  Layouts tested per agent : 30")
    print("  Seeds used               : 1000 .. 1029")

    print("\n--- AVERAGE STEPS PER LAYOUT ---")
    print(f"  Simple reflex (current cell only) : {avg_reflex:.2f}")
    print(f"  Sensor-upgraded reflex            : {avg_sensor:.2f}")
    print(f"  Improvement                       : {avg_reflex - avg_sensor:.2f}")

                 LAB TASK 6: SENSOR UPGRADE                 

--- CONFIGURATION ---
  Layouts tested per agent : 30
  Seeds used               : 1000 .. 1029

--- AVERAGE STEPS PER LAYOUT ---
  Simple reflex (current cell only) : 75.93
  Sensor-upgraded reflex            : 32.23
  Improvement                       : 43.70


In [6]:
import random


# ============================================================
# Pretty Output Helpers
# ============================================================
def print_banner(title):
    width = 60
    print("=" * width)
    print(title.center(width))
    print("=" * width)


# ============================================================
# Environments
# ============================================================
class GridEnvironment:
    def __init__(self, rows=4, cols=4, dirt_probability=0.5, seed=None):
        if seed is not None:
            random.seed(seed)
        self.rows, self.cols = rows, cols
        self.dirt = [
            [random.random() < dirt_probability for _ in range(cols)]
            for _ in range(rows)
        ]
        self.position = (0, 0)

    def get_percept(self):
        r, c = self.position
        return {"position": (r, c), "is_dirty": self.dirt[r][c]}

    def valid_moves(self):
        r, c = self.position
        moves = []
        if c < self.cols - 1:
            moves.append("Right")
        if c > 0:
            moves.append("Left")
        if r < self.rows - 1:
            moves.append("Down")
        if r > 0:
            moves.append("Up")
        return moves

    def apply_action(self, action):
        r, c = self.position
        if action == "Suck":
            self.dirt[r][c] = False
        elif action == "Right" and c < self.cols - 1:
            self.position = (r, c + 1)
        elif action == "Left" and c > 0:
            self.position = (r, c - 1)
        elif action == "Down" and r < self.rows - 1:
            self.position = (r + 1, c)
        elif action == "Up" and r > 0:
            self.position = (r - 1, c)

    def is_clean(self):
        return not any(any(row) for row in self.dirt)


class ValuedGridEnvironment(GridEnvironment):
    def __init__(self, rows=4, cols=4, dirt_probability=0.5, seed=None):
        super().__init__(rows, cols, dirt_probability, seed)
        self.value = [
            [random.randint(1, 5) for _ in range(cols)]
            for _ in range(rows)
        ]

    def get_percept(self):
        return {
            "position": self.position,
            "is_dirty": self.dirt[self.position[0]][self.position[1]],
            "dirt_grid": [row[:] for row in self.dirt],
            "value_grid": [row[:] for row in self.value],
        }


# ============================================================
# Helpers + Agent
# ============================================================
def manhattan(a, b):
    return abs(a[0] - b[0]) + abs(a[1] - b[1])


def goal_based_agent(position, goal):
    r, c = position
    gr, gc = goal
    if r < gr:
        return "Down"
    if r > gr:
        return "Up"
    if c < gc:
        return "Right"
    if c > gc:
        return "Left"
    return "NoOp"


def utility_agent(percept, env, use_distance=True):
    r, c = percept["position"]
    dirty_cells = [
        (i, j) for i in range(env.rows) for j in range(env.cols)
        if percept["dirt_grid"][i][j]
    ]
    if not dirty_cells:
        return "NoOp"
    if percept["is_dirty"]:
        return "Suck"

    def score(cell):
        val = percept["value_grid"][cell[0]][cell[1]]
        dist = manhattan((r, c), cell)
        return val - dist if use_distance else val

    target = max(dirty_cells, key=score)
    return goal_based_agent((r, c), target)


# ============================================================
# Main Run
# ============================================================
if __name__ == "__main__":
    print_banner("LAB TASK 7: UTILITY-BASED AGENT")

    # --- Self-check on a fixed layout ---
    env = ValuedGridEnvironment(rows=4, cols=4, dirt_probability=0.5, seed=42)
    env.position = (0, 0)
    env.dirt = [
        [True, True, False, False],
        [False, False, False, False],
        [False, False, False, False],
        [False, False, False, False],
    ]
    env.value = [
        [5, 1, 1, 1],
        [1, 1, 1, 1],
        [1, 1, 1, 1],
        [1, 1, 1, 1],
    ]
    print("\n--- SELF-CHECK ---")
    print(f"  Self-check action: {utility_agent(env.get_percept(), env)}")

    # --- Compare value-alone vs value-minus-distance ---
    def run_utility(use_distance, n=30):
        totals = []
        for i in range(n):
            env = ValuedGridEnvironment(rows=4, cols=4, dirt_probability=0.5,
                                        seed=500 + i)
            value_cleaned = 0
            for _ in range(20):
                p = env.get_percept()
                if not any(any(row) for row in p["dirt_grid"]):
                    break
                a = utility_agent(p, env, use_distance=use_distance)
                if a == "Suck":
                    value_cleaned += env.value[p["position"][0]][p["position"][1]]
                env.apply_action(a)
            totals.append(value_cleaned)
        return sum(totals) / len(totals)

    print("\n--- AVERAGE CLEANED VALUE OVER 30 LAYOUTS ---")
    print(f"  Value alone            : {run_utility(False):.2f}")
    print(f"  Value minus distance   : {run_utility(True):.2f}")

              LAB TASK 7: UTILITY-BASED AGENT               

--- SELF-CHECK ---
  Self-check action: Suck

--- AVERAGE CLEANED VALUE OVER 30 LAYOUTS ---
  Value alone            : 21.00
  Value minus distance   : 21.80


In [7]:
import random


# ============================================================
# Pretty Output Helpers
# ============================================================
def print_banner(title):
    width = 60
    print("=" * width)
    print(title.center(width))
    print("=" * width)


def print_counts(counts):
    print("\n--- OUTCOME COUNTS ---")
    print(f"  done    : {counts['done']}")
    print(f"  refused : {counts['refused']}")
    print(f"  blocked : {counts['blocked']}")


def print_log_table(entries, title):
    print(f"\n--- {title} ---")
    print(f"  {'step':<6}{'position':<12}{'action':<10}{'outcome':<10}")
    print("  " + "-" * 38)
    for e in entries:
        print(f"  {e['step']:<6}{str(e['position']):<12}"
              f"{e['action']:<10}{e['outcome']:<10}")


# ============================================================
# Base Environment
# ============================================================
class GridEnvironment:
    def __init__(self, rows=4, cols=4, dirt_probability=0.5, seed=None):
        if seed is not None:
            random.seed(seed)
        self.rows, self.cols = rows, cols
        self.dirt = [
            [random.random() < dirt_probability for _ in range(cols)]
            for _ in range(rows)
        ]
        self.position = (0, 0)

    def get_percept(self):
        r, c = self.position
        return {"position": (r, c), "is_dirty": self.dirt[r][c]}

    def valid_moves(self):
        r, c = self.position
        moves = []
        if c < self.cols - 1:
            moves.append("Right")
        if c > 0:
            moves.append("Left")
        if r < self.rows - 1:
            moves.append("Down")
        if r > 0:
            moves.append("Up")
        return moves

    def apply_action(self, action):
        r, c = self.position
        if action == "Suck":
            self.dirt[r][c] = False
        elif action == "Right" and c < self.cols - 1:
            self.position = (r, c + 1)
        elif action == "Left" and c > 0:
            self.position = (r, c - 1)
        elif action == "Down" and r < self.rows - 1:
            self.position = (r + 1, c)
        elif action == "Up" and r > 0:
            self.position = (r - 1, c)

    def is_clean(self):
        return not any(any(row) for row in self.dirt)


# ============================================================
# Helpers + Restricted Rooms
# ============================================================
def neighbor_position(pos, action):
    r, c = pos
    return {
        "Right": (r, c + 1),
        "Left": (r, c - 1),
        "Down": (r + 1, c),
        "Up": (r - 1, c),
    }[action]


RESTRICTED = {(1, 1), (2, 2)}


# ============================================================
# Guarded Environment
# ============================================================
class GuardedEnvironment(GridEnvironment):
    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.log = []
        self.counts = {"done": 0, "refused": 0, "blocked": 0}

    def valid_moves(self):
        base = super().valid_moves()
        r, c = self.position
        return [m for m in base
                if neighbor_position((r, c), m) not in RESTRICTED]

    def apply_action(self, action):
        r, c = self.position
        if action in ("Right", "Left", "Down", "Up"):
            nr, nc = neighbor_position((r, c), action)
            if (nr, nc) in RESTRICTED:
                self.counts["blocked"] += 1
                self.log.append({"step": len(self.log), "position": (r, c),
                                 "action": action, "outcome": "blocked"})
                return
        super().apply_action(action)
        self.log.append({"step": len(self.log), "position": (r, c),
                         "action": action, "outcome": "done"})
        self.counts["done"] += 1


# ============================================================
# Human Approval Hook + Loop + Agent
# ============================================================
def ask_human(room):
    return False


def run_agent(env, decide, max_steps=200):
    steps = 0
    for _ in range(max_steps):
        percept = env.get_percept()
        if env.is_clean():
            break
        action = decide(percept, env)
        env.apply_action(action)
        steps += 1
    return steps


def guarded_agent(percept, env):
    r, c = percept["position"]
    if percept["is_dirty"]:
        if (r, c) in RESTRICTED or (r, c) in {(1, 0), (0, 1), (2, 1), (1, 2)}:
            if not ask_human((r, c)):
                env.counts["refused"] += 1
                env.log.append({"step": len(env.log), "position": (r, c),
                                "action": "Suck", "outcome": "refused"})
                return (random.choice(env.valid_moves())
                        if env.valid_moves() else "NoOp")
        return "Suck"
    return random.choice(env.valid_moves()) if env.valid_moves() else "NoOp"


# ============================================================
# Main Run
# ============================================================
if __name__ == "__main__":
    print_banner("LAB TASK 8: GUARDRAILS IN CODE")
    print("Restricted rooms (least access):", RESTRICTED)
    print("Human approval hook returns    : False (always refuse)")
    print("Seed                           : 11")
    print("Max steps                      : 60")

    env = GuardedEnvironment(rows=4, cols=4, dirt_probability=0.5, seed=11)
    steps_taken = run_agent(env, guarded_agent, max_steps=60)

    print(f"\nTotal steps executed: {steps_taken}")
    print_counts(env.counts)

    print_log_table(env.log[:5], "FIRST 5 LOG ENTRIES")
    print_log_table(env.log[-5:], "LAST 5 LOG ENTRIES")

    assert env.counts["refused"] >= 1, "Expected at least one refusal"
    print("\n[OK] Assert passed: at least one refusal was recorded.")

               LAB TASK 8: GUARDRAILS IN CODE               
Restricted rooms (least access): {(1, 1), (2, 2)}
Human approval hook returns    : False (always refuse)
Seed                           : 11
Max steps                      : 60

Total steps executed: 60

--- OUTCOME COUNTS ---
  done    : 60
  refused : 3
  blocked : 0

--- FIRST 5 LOG ENTRIES ---
  step  position    action    outcome   
  --------------------------------------
  0     (0, 0)      Suck      done      
  1     (0, 0)      Down      done      
  2     (1, 0)      Up        done      
  3     (0, 0)      Right     done      
  4     (0, 1)      Right     done      

--- LAST 5 LOG ENTRIES ---
  step  position    action    outcome   
  --------------------------------------
  58    (3, 0)      Right     done      
  59    (3, 1)      Up        done      
  60    (2, 1)      Left      done      
  61    (2, 0)      Up        done      
  62    (1, 0)      Up        done      

[OK] Assert passed: at least one refu